# Homework 8: curve fitting, objects and minimizing

Covers **Lecture 19** (`curve_fit`, `p0`, standard errors, residuals), **Lecture 20** (objects, attributes and methods, `dir`, the `linregress` result) and **Lecture 21** (`minimize`, `x0`, the result object, inversion by squaring).
Functions from Lectures 17 and 18 appear as a supporting tool.

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning.
Reloading or disconnecting can revert the notebook to the instructor copy.

# Problem 0 (-10 pts if blank): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work. Critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

# Dataset

A steel is held at a fixed temperature while a new phase grows through it.
A lab records the fraction of the sample that has transformed, from 0 (none) to 1 (all), at 28 different times.

The Johnson-Mehl-Avrami-Kolmogorov (JMAK) equation describes this kind of transformation:

$$X = 1 - e^{-k t^n}$$

Here $t$ is the time in minutes, $k$ is a rate constant in min$^{-n}$, and $n$ has no units.

The cell below reads the table and pulls each column out as an array, as in Lecture 19.

*Data: synthetic, generated from the JMAK equation with measurement noise added.*

In [1]:
import numpy as np
import pandas as pd

avrami = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/avrami_transformation.csv')

time_min = avrami['time_min'].to_numpy()                    # min
fraction = avrami['fraction_transformed'].to_numpy()        # unitless, 0 to 1

print(len(time_min), 'readings')

28 readings


`time_min` is the time of each reading, in min, and `fraction` is the fraction transformed at that time.

# Problem 1 (35 pts): `curve_fit`, `p0`, and standard errors

(a) Write a function `jmak(t, k, n)` that returns the fraction transformed, $1 - e^{-k t^n}$.

- Write a docstring that gives the unit of each parameter and of the result.
- Print `jmak(5, 0.015, 2.5)`.

In [2]:
# your code here

(b) Fit `jmak` to the readings with `curve_fit`.

- Import `optimize` from `scipy`.
- Pass `jmak`, `time_min` and `fraction`, with `p0=[0.01, 2]`, and unpack the result as `popt, pcov`.
- Compute `perr` from `pcov` in one line.
- Print `k` as value ± SE with its unit.
- Print `n` as value ± SE.

> Pass `jmak` without parentheses. `popt` follows the order in the `def` line. Round each SE to one or two digits, and give the value to the same decimal place.

In [3]:
# your code here

(c) Run the cell below to define `r_squared`, the same function as in Lecture 19.

In [4]:
def r_squared(y, y_fit):
    """R2 of a fit: 1 minus the squared misfit over the squared spread of y."""
    return 1 - np.sum((y - y_fit)**2) / np.sum((y - np.mean(y))**2)

Check the fit.

- Compute `fraction_fit` by calling `jmak` on `time_min` with the fitted values.
- Print `r_squared(fraction, fraction_fit)`.
- Plot the readings as dots and the fitted curve as a line, with both axes labeled with units.
- Plot the residuals, `fraction - fraction_fit`, against `time_min`, with a black line at zero.

> Use `np.linspace(1.5, 8.5, 100)` for the smooth curve.

In [5]:
# your code here

(d) Fit again from a poor starting guess.

- Call `curve_fit` with `p0=[50, 0.001]` and unpack the result as `popt_bad, pcov_bad`.
- Print `popt_bad`.
- Print `r_squared` for the fitted curve from `popt_bad`.

Then answer in two short bullets.

- Do the residuals from part (c) scatter evenly around zero, or form a pattern?
- What three signs show that the poor start did not give a useful fit?

> Look for starting values returned unchanged, a negative R², and a warning about the covariance.

In [6]:
# your code here

*your answer here*

# Problem 2 (20 pts): attributes, methods, `dir`, and the `linregress` result

(a) The table `avrami` is an object.

- Print `type(avrami)`.
- Print `avrami.shape`.
- Print `avrami.head(3)`.
- Add a comment on each `print` saying whether the name after the dot is an attribute or a method.

In [7]:
# your code here

(b) The Hall-Petch relation makes the yield strength of copper a straight line in $d^{-1/2}$, where $d$ is the grain diameter.

The cell below reads the table and keeps two columns.

In [8]:
hp = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/hall_petch.csv')

inv_sqrt_d = hp['inv_sqrt_grain_size']      # m^-1/2
yield_mpa = hp['yield_strength_mpa']        # MPa

Fit the line with `stats.linregress`.

- Import `stats` from `scipy`.
- Store the result as `fit` and print `type(fit)`.
- Print `'stderr' in dir(fit)`.
- Print `'rvalue' in dir(fit)`.
- Print the slope as value ± SE, rounded to three decimals, with `fit.slope` and `fit.stderr`.
- Print R² from `fit.rvalue`.

In [9]:
# your code here

(c) Use the slope and its standard error.

- Print the rise in yield strength, in MPa, when $d^{-1/2}$ grows by 100 m$^{-1/2}$, as value ± SE.
- In one or two sentences, say why `fit.slope` is written without parentheses.

In [10]:
# your code here

*your answer here*

# Problem 3 (20 pts): an objective function and `minimize`

The Lennard-Jones model gives the energy of two argon atoms at a separation $r$:

$$E(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

In [11]:
epsilon = 0.0103   # well depth, eV
sigma = 3.40      # separation where E crosses zero, Å

(a) Write a function `pair_energy(r)` that returns $E(r)$ in eV, for a separation `r` in Å.

- Read `epsilon` and `sigma` from the notebook, as in Lecture 21.
- Write a docstring that gives the unit of `r` and of the result.
- Print `pair_energy(3.40)`.
- Print `pair_energy(4.0)`.

In [12]:
# your code here

(b) Plot `pair_energy` for `r` from 3.2 Å to 8 Å.

- Use `np.linspace(3.2, 8, 200)`.
- Draw a dashed black line at zero with `ax.axhline`.
- Label both axes with units.

> Read a starting value for `x0` off this plot in part (c).

In [13]:
# your code here

(c) Find the lowest point with `minimize`.

- Call `optimize.minimize(pair_energy, 4.0)` and store the result as `res`.
- Print `res.x[0]` with `Å`.
- Print `res.fun` with `eV`.
- Print `res.success`.

In [14]:
# your code here

(d) The result `res.fun` is negative.

- In one or two sentences, say what the value of `res.fun` means for the pair, in eV.

*your answer here*

# Problem 4 (25 pts): inverting a fit with `minimize`

A heat treater needs the time to reach a chosen fraction transformed, so these parts run the fitted curve backward.

In [15]:
k_fit = 0.0155   # min^-n, from Problem 1
n_fit = 2.47     # unitless, from Problem 1

(a) Write a function `fraction_model(t)` that returns the fraction transformed from the fitted values above.

- Read `k_fit` and `n_fit` from the notebook.
- Write a docstring that gives the unit of `t` and of the result.
- Set `target = 0.5` on its own line.
- Plot `fraction_model` for `t` from 0 to 10 min, with a dashed black line at `target` and both axes labeled with units.

> Use `np.linspace(0, 10, 200)`. A point where the curve meets the dashed line has the time you want.

In [16]:
# your code here

(b) Write an objective function `miss(t)` that returns `((fraction_model(t) - target) / target)**2`.

- Call `optimize.minimize(miss, 5)` and store the result as `res`.
- Print `res.x[0]` with `min`.
- Print `res.fun`.
- Print `res.success`.
- Print `fraction_model(res.x[0])` to check the answer.

> A correct inverse has `res.fun` near zero.

In [17]:
# your code here

(c) Find the time to reach 90% transformed.

- Set `target = 0.9` on its own line, and do not edit `miss`.
- Call `optimize.minimize(miss, 5)` and print `res.x[0]` with `min`.
- Print `res.success`.

> `miss` reads `target` from the notebook when it is called.

In [18]:
# your code here

(d) Set `target = 0.5` again and call `optimize.minimize(miss, 20)`.

- Print `res.x[0]`.
- Print `res.fun`.
- Print `res.success`.
- In one or two sentences, explain why the value of `res.fun` means the answer is not the time to reach 50%, even though `res.success` is `True`.

In [19]:
# your code here

*your answer here*